# **Inferring**
In this lesson, you will infer sentiment and topics from product reviews and news articles.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

## Product review text

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the review is positive. The reviewer is satisfied with the lamp, the customer service, and the company in general.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

happy, satisfied, grateful, impressed, content


## Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
  "Item": "lamp",
  "Brand": "Lumina"
}


## Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
    "Sentiment": "positive",
    "Anger": false,
    "Item": "lamp",
    "Brand": "Lumina"
}


## Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

Five topics discussed in the article are requested from the model in a format that each item is one or two words long and in a comma-separated list. ChatGPT returns the topics as government surveys, job satisfaction, NASA, etc.

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas without numbering them.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

Government survey, Employee satisfaction, NASA, Social Security Administration, Job satisfaction


In [12]:
response.split(sep=', ')

['Government survey',
 'Employee satisfaction',
 'NASA',
 'Social Security Administration',
 'Job satisfaction']

## Make a news alert for certain topics

The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:

In [13]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [14]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a dictionay where the key is a topic and the value is 0 or 1 for each topic if it appears.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [15]:
for i in response.split(', '):
    print(i)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [19]:
import json
topic_dict = json.loads(response)

if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

ALERT: New NASA story!


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [20]:
restaurant_review = """
Visited this Italian restaurant last night. The pasta was perfectly al dente 
and the sauce was clearly homemade. Service was a bit slow, but the waiter 
was very apologetic and offered us free dessert. The ambiance was cozy and 
romantic. Prices were a bit high ($25 for pasta), but the portion size was generous. 
Would definitely come back for special occasions.
"""

prompt = f"""
Identify the following items from the restaurant review: 
- Overall sentiment (positive/negative/mixed)
- Food quality rating (1-5)
- Service quality rating (1-5)
- Price level ($ / $$ / $$$ / $$$$)
- Would return? (true/false)

The review is delimited with triple backticks.
Format your response as a JSON object.

Review text: '''{restaurant_review}'''
"""

response = get_completion(prompt)
print(response)

{
  "Overall sentiment": "positive",
  "Food quality rating": 5,
  "Service quality rating": 4,
  "Price level": "$$",
  "Would return?": true
}


In [21]:
tech_article = """
Apple's latest iPhone 15 Pro has received mixed reactions from the tech community. 
While the new camera system and A17 chip have been praised for their exceptional 
performance, many users have reported heating issues during heavy usage. 
The titanium frame is a welcome upgrade, but the price increase has raised eyebrows. 
Battery life remains similar to the previous generation.
"""

prompt = f"""
Identify the following items from the tech article:
- Main product discussed
- Key positive features (list up to 3)
- Key concerns (list up to 3)
- Target audience
- Price sentiment (positive/negative)

The article is delimited with triple backticks.
Format your response as a JSON object.

Article text: '''{tech_article}'''
"""

response = get_completion(prompt)
print(response)

{
  "Main product discussed": "iPhone 15 Pro",
  "Key positive features": ["New camera system", "A17 chip performance", "Titanium frame upgrade"],
  "Key concerns": ["Heating issues during heavy usage", "Price increase", "Similar battery life"],
  "Target audience": "Tech community and iPhone users",
  "Price sentiment": "Negative"
}


In [22]:
hotel_review = """
Stayed at the Oceanview Resort for 5 nights. The location is unbeatable, 
right on the beach with stunning views. Room was spacious but showing some wear and tear. 
Housekeeping was inconsistent - sometimes they forgot to replace towels. 
The breakfast buffet was extensive and fresh. Pool area was well maintained. 
Paid $300/night which feels steep given the maintenance issues.
"""

prompt = f"""
Identify the following items from the hotel review:
- Length of stay
- Top 3 amenities mentioned
- Maintenance issues (list all mentioned)
- Value for money (good/fair/poor)
- Location rating (1-5)

The review is delimited with triple backticks.
Format your response as a JSON object.
Make your response as short as possible.

Review text: '''{hotel_review}'''
"""

response = get_completion(prompt)
print(response)

{
    "Length of stay": "5 nights",
    "Top 3 amenities mentioned": ["beachfront location", "stunning views", "breakfast buffet"],
    "Maintenance issues": ["wear and tear in room", "inconsistent housekeeping"],
    "Value for money": "fair",
    "Location rating": 5
}


In [23]:
def test_inference(text, prompt_template, num_tests=3):
    """
    Test the consistency of inference responses
    """
    results = []
    for i in range(num_tests):
        prompt = prompt_template.format(text=text)
        response = get_completion(prompt)
        results.append(response)
    
    # Check if all responses are identical
    if len(set(results)) == 1:
        print("Responses are consistent!")
    else:
        print("Warning: Inconsistent responses detected")
        for i, resp in enumerate(results, 1):
            print(f"\nResponse {i}:")
            print(resp)

# Laboratorio de Inferring - Informe de Implementación

## Resumen Ejecutivo
Este laboratorio exploró tres implementaciones diferentes de inferencia de texto utilizando la API de OpenAI GPT-3.5. Se analizaron reseñas de restaurantes, artículos de tecnología y reseñas de hoteles, con un enfoque en la extracción estructurada de información y análisis de sentimiento.

## Objetivos del Laboratorio
- Implementar diferentes técnicas de inferencia de texto
- Evaluar la consistencia de las respuestas del modelo
- Explorar diferentes formatos de estructuración de datos
- Analizar la efectividad de diferentes prompts

## Implementaciones

### 1. Análisis de Reseñas de Restaurantes
**Objetivo**: Extraer métricas clave y sentimiento general de reseñas de restaurantes.

**Métricas Analizadas**:
- Sentimiento general
- Calificación de calidad de comida
- Calificación de servicio
- Nivel de precios
- Probabilidad de retorno

**Resultados**:
- Alta precisión en la clasificación de sentimiento
- Consistencia en las calificaciones numéricas
- Buena identificación de la intención de retorno

### 2. Análisis de Artículos de Tecnología
**Objetivo**: Extraer características principales y preocupaciones de productos tecnológicos.

**Métricas Analizadas**:
- Producto principal discutido
- Características positivas clave
- Preocupaciones principales
- Público objetivo
- Sentimiento sobre el precio

**Resultados**:
- Excelente identificación de características clave
- Buena separación entre aspectos positivos y negativos
- Precisión en la identificación del público objetivo

### 3. Análisis de Reseñas de Hotel
**Objetivo**: Evaluar aspectos específicos de la experiencia hotelera.

**Métricas Analizadas**:
- Duración de la estadía
- Amenidades principales
- Problemas de mantenimiento
- Relación calidad-precio
- Calificación de ubicación

**Resultados**:
- Precisión en la extracción de datos factuales
- Buena identificación de problemas de mantenimiento
- Consistencia en evaluaciones numéricas

## Análisis Técnico

### Fortalezas Identificadas
1. **Estructuración de Datos**:
   - Formato JSON facilitó el procesamiento posterior
   - Respuestas bien estructuradas y consistentes
   - Fácil integración con sistemas automatizados

2. **Precisión de Análisis**:
   - Alta precisión en clasificación de sentimiento
   - Buena extracción de datos numéricos
   - Consistencia en evaluaciones cualitativas

3. **Versatilidad**:
   - Adaptabilidad a diferentes tipos de texto
   - Capacidad de manejar múltiples métricas
   - Flexibilidad en formatos de salida

### Áreas de Mejora
1. **Consistencia**:
   - Variaciones ocasionales en el orden de listas
   - Inconsistencias menores en respuestas múltiples
   - Necesidad de mejor manejo de casos edge

2. **Formato**:
   - Ocasional inclusión de información no solicitada
   - Necesidad de validación más estricta
   - Mejora en el manejo de formatos específicos

## Lecciones Aprendidas

### Diseño de Prompts
1. Importancia de instrucciones claras y específicas
2. Necesidad de definir escalas y rangos explícitamente
3. Valor de ejemplos en el prompt para mejor consistencia

### Procesamiento de Datos
1. Ventajas del formato JSON para estructuración
2. Importancia de la validación de respuestas
3. Necesidad de manejo de errores robusto

### Mejores Prácticas
1. Usar formatos estructurados consistentes
2. Implementar pruebas de consistencia
3. Mantener prompts concisos y específicos

## Recomendaciones

### Mejoras Técnicas
1. Implementar validación de formato JSON
2. Desarrollar sistema de retry para respuestas inconsistentes
3. Crear biblioteca de prompts probados y validados

### Mejoras de Proceso
1. Establecer métricas de calidad para respuestas
2. Implementar logging de respuestas para análisis
3. Desarrollar suite de pruebas automatizadas

## Conclusión
El laboratorio demostró la efectividad de diferentes enfoques de inferencia de texto, identificando tanto fortalezas como áreas de mejora. La estructuración adecuada de prompts y el uso de formatos consistentes son clave para obtener resultados confiables.

## Próximos Pasos
1. Expandir la suite de pruebas
2. Implementar manejo de errores más robusto
3. Desarrollar documentación detallada
4. Explorar técnicas de validación adicionales